[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def clip_grad_norm(parameters, max_norm):

    # Gradient가 존재하는 파라미터만 추출 --> [Grad-None param 발생 경우] -->
    # (1) Parameter Freezing, (2) Loss 계산 참여 X, (3) .detach() 또는 .no_grad()로 계산 그래프 연결 차단
    parameters = [p for p in parameters if p.grad is not None]

    if not parameters:
        return 0.0

    # 전체 Gradient의 L2 Norm 계산
    total_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))

    # Clipping 계수 계산
    clip_coef = max_norm / (total_norm + 1e-6)

    # 전체 Norm이 max_norm을 초과하면 Gradient 축소
    if clip_coef < 1:
        with torch.no_grad():
            for p in parameters:
                p.grad.mul_(clip_coef)  # in-place multiplication (_)

    return total_norm.item()

In [11]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()

print('Before:', p.grad.norm().item())
print(p.grad)
print()

orig = clip_grad_norm([p], max_norm=1.0)

print('After: ', p.grad.norm().item())
print(p.grad)
print('Original norm:', orig)

Before: 100.0
tensor([10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
        10., 10.])

After:  0.9999999403953552
tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0

In [5]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Clips to max_norm (2.5ms)
  ✅ [2/4] Returns original norm (0.7ms)
  ✅ [3/4] No change when norm < max_norm (1.7ms)
  ✅ [4/4] Preserves direction (20.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (25.2ms total)
  Progress saved. Run status() to see your dashboard.



In [7]:
p = torch.tensor([1.0, 2.0], requires_grad=True)

# Gradient가 [3, 4]가 되도록 Loss 정의
loss = 3 * p[0] + 4 * p[1]
loss.backward()

print("Before:", p.grad)
print("Before Norm:", p.grad.norm().item())
print()

original_norm = clip_grad_norm([p], max_norm=1.0)

print("After:", p.grad)
print("After Norm:", p.grad.norm().item())
print("Original Norm:", original_norm)

Before: tensor([3., 4.])
Before Norm: 5.0

After: tensor([0.6000, 0.8000])
After Norm: 0.9999998211860657
Original Norm: 5.0


In [12]:
import torch

# 1. Create parameters
W = torch.tensor([[1., 2.], [3., 4.]], requires_grad=True)  # 2D
b = torch.tensor([0.5, -0.5], requires_grad=True)            # 1D
c = torch.tensor(2.0, requires_grad=True)                    # 0D
unused = torch.tensor([10., 20.], requires_grad=True)        # 1D

named_params = {
    "W (matrix)": W,
    "b (vector)": b,
    "c (scalar)": c,
    "unused": unused
}

parameters = list(named_params.values())


# 2. Before backward
print("=== Before backward ===")

for name, p in named_params.items():
    print(f"{name}: shape={tuple(p.shape)}, grad={p.grad}")


# 3. Compute loss & backward
loss = (
    (W * torch.tensor([[3., 4.], [0., 12.]])).sum()
    + (b * torch.tensor([0., 0.])).sum()
    + 5 * c
)

loss.backward()


# 4. Inspect gradients
print("\n=== After backward ===")

for name, p in named_params.items():
    print(f"\n{name}")
    print(f"  Parameter shape: {tuple(p.shape)}")
    print(f"  Gradient: {p.grad}")
    print(f"  Gradient Norm: {p.grad.norm().item() if p.grad is not None else None}")


# 5. Compute total gradient norm
grads = [p.grad for p in parameters if p.grad is not None]

total_norm = torch.sqrt(sum(g.norm() ** 2 for g in grads))

print("\n=== Total Gradient Norm ===")
print(f"Number of parameters: {len(parameters)}")
print(f"Number of valid gradients: {len(grads)}")
print(f"Total Norm: {total_norm.item():.4f}")


# 6. Apply gradient clipping
original_norm = clip_grad_norm(parameters, max_norm=5.0)

print("\n=== After Clipping (max_norm=5.0) ===")
print(f"Original Norm: {original_norm:.4f}")

for name, p in named_params.items():
    print(f"{name}: grad={p.grad}")

new_norm = torch.sqrt(sum(g.norm() ** 2 for g in grads))
print(f"\nClipped Total Norm: {new_norm.item():.4f}")

=== Before backward ===
W (matrix): shape=(2, 2), grad=None
b (vector): shape=(2,), grad=None
c (scalar): shape=(), grad=None
unused: shape=(2,), grad=None

=== After backward ===

W (matrix)
  Parameter shape: (2, 2)
  Gradient: tensor([[ 3.,  4.],
        [ 0., 12.]])
  Gradient Norm: 13.0

b (vector)
  Parameter shape: (2,)
  Gradient: tensor([0., 0.])
  Gradient Norm: 0.0

c (scalar)
  Parameter shape: ()
  Gradient: 5.0
  Gradient Norm: 5.0

unused
  Parameter shape: (2,)
  Gradient: None
  Gradient Norm: None

=== Total Gradient Norm ===
Number of parameters: 4
Number of valid gradients: 3
Total Norm: 13.9284

=== After Clipping (max_norm=5.0) ===
Original Norm: 13.9284
W (matrix): grad=tensor([[1.0769, 1.4359],
        [0.0000, 4.3077]])
b (vector): grad=tensor([0., 0.])
c (scalar): grad=1.7948951721191406
unused: grad=None

Clipped Total Norm: 5.0000


In [13]:
import torch
import torch.nn as nn

# Experimental settings
x = torch.tensor([[10., 1.]])
target = torch.tensor([[0.]])

lr = 0.1
max_norm = 1.0


def run_experiment(use_clipping):

    # 1. Initialize model
    model = nn.Linear(2, 1, bias=False)

    with torch.no_grad():
        model.weight.copy_(torch.tensor([[1., 0.]]))

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # 2. Forward pass
    optimizer.zero_grad()

    prediction = model(x)
    loss = ((prediction - target) ** 2).mean()

    loss_before = loss.item()

    # 3. Backward pass
    loss.backward()

    grad_before = model.weight.grad.detach().clone()
    norm_before = grad_before.norm().item()

    # 4. Gradient clipping (optional)
    if use_clipping:
        original_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_norm
        )

    grad_after = model.weight.grad.detach().clone()
    norm_after = grad_after.norm().item()

    # 5. Update weights
    optimizer.step()

    # 6. Evaluate updated model
    with torch.no_grad():
        loss_after = ((model(x) - target) ** 2).mean().item()
        weight_after = model.weight.detach().clone()

    # 7. Print results
    print(f"\n=== Clipping: {use_clipping} ===")
    print(f"Loss Before: {loss_before:.4f}")
    print(f"Gradient Before: {grad_before.flatten().tolist()}")
    print(f"Gradient Norm Before: {norm_before:.4f}")

    print(f"Gradient After: {grad_after.flatten().tolist()}")
    print(f"Gradient Norm After: {norm_after:.4f}")

    print(f"Updated Weights: {weight_after.flatten().tolist()}")
    print(f"Loss After: {loss_after:.4f}")


# Compare two experiments
run_experiment(use_clipping=False)
run_experiment(use_clipping=True)


=== Clipping: False ===
Loss Before: 100.0000
Gradient Before: [200.0, 20.0]
Gradient Norm Before: 200.9975
Gradient After: [200.0, 20.0]
Gradient Norm After: 200.9975
Updated Weights: [-19.0, -2.0]
Loss After: 36864.0000

=== Clipping: True ===
Loss Before: 100.0000
Gradient Before: [200.0, 20.0]
Gradient Norm Before: 200.9975
Gradient After: [0.9950371384620667, 0.09950371086597443]
Gradient Norm After: 1.0000
Updated Weights: [0.9004963040351868, -0.009950371459126472]
Loss After: 80.9102
